# Laporan Jobsheet 09: Praktikum 3 - Klasifikasi Naive Bayes
**Mata Kuliah:** Pembelajaran Mesin  
**Jurusan / Program Studi:** Teknologi Informasi / D4 Teknik Informatika - POLINEMA  
**Dataset:** Sleep Health and Lifestyle Dataset  
**Topik:** Praktikum 3 - Klasifikasi Naive Bayes (MultinomialNB & GaussianNB)


---
# 🥕 Praktikum 3: Klasifikasi Naive Bayes dengan Dataset Sleep and Health

Pada praktikum ini, kita mengimplementasikan algoritma Naive Bayes untuk melakukan klasifikasi status gangguan tidur (*Sleep Disorder*: None, Insomnia, Sleep Apnea). Kita mengikuti struktur tahapan Praktikum 3 dari modul JS09:
- **Langkah 1:** Load Data
- **Langkah 2:** Preprocessing (Drop kolom ID, penanganan missing values, ekstraksi fitur tekanan darah, encoding label & kategorikal)
- **Langkah 3:** Pemisahan Fitur dan Target serta Split Data Train-Test (80:20)
- **Langkah 4:** Training dan Evaluasi Model (MultinomialNB & GaussianNB)


In [ ]:
# Langkah 1 - Load Data
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Membaca dataset
df = pd.read_csv('Sleep_health_and_lifestyle_dataset.csv')

# Menampilkan 5 data teratas
df.head()


In [ ]:
# Langkah 2 - Preprocessing

# 2a. Penanganan nilai label 'None' (pandas membacanya sebagai NaN secara default)
df['Sleep Disorder'] = df['Sleep Disorder'].fillna('None')

# 2b. Drop kolom 'Person ID' yang tidak digunakan untuk permodelan
df = df.drop(columns=['Person ID'])

# 2c. Memecah fitur 'Blood Pressure' menjadi 'Systolic_BP' dan 'Diastolic_BP'
df[['Systolic_BP', 'Diastolic_BP']] = df['Blood Pressure'].str.split('/', expand=True).astype(int)
df = df.drop(columns=['Blood Pressure'])

# Standarisasi nilai 'Normal Weight' menjadi 'Normal' pada kolom BMI Category
df['BMI Category'] = df['BMI Category'].replace({'Normal Weight': 'Normal'})

# 2d. Inspeksi Data
print("Distribusi Target (Sleep Disorder):")
print(df['Sleep Disorder'].value_counts())
print('\nInformasi Dataset:')
print(df.info())
print('\nStatistik Deskriptif:')
display(df.describe())


In [ ]:
# 2e. Encoding Fitur Kategorikal dan Label Target
from sklearn.preprocessing import LabelEncoder

le_gender = LabelEncoder()
le_occ = LabelEncoder()
le_bmi = LabelEncoder()
le_target = LabelEncoder()

df_encoded = df.copy()
df_encoded['Gender'] = le_gender.fit_transform(df_encoded['Gender'])
df_encoded['Occupation'] = le_occ.fit_transform(df_encoded['Occupation'])
df_encoded['BMI Category'] = le_bmi.fit_transform(df_encoded['BMI Category'])
df_encoded['Sleep Disorder'] = le_target.fit_transform(df_encoded['Sleep Disorder'])

print("Mapping Label Target:")
for idx, label in enumerate(le_target.classes_):
    print(f"{label} -> {idx}")

df_encoded.head()


In [ ]:
# Langkah 3 - Pemisahan Fitur & Target serta Data Splitting
from sklearn.model_selection import train_test_split

# Memisahkan Fitur (X) dan Target (y)
X = df_encoded.drop(columns=['Sleep Disorder'])
y = df_encoded['Sleep Disorder']

# Split Data (80% Train, 20% Test) dengan stratifikasi agar proporsi kelas seimbang
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Jumlah sampel Train : {X_train.shape[0]}")
print(f"Jumlah sampel Test  : {X_test.shape[0]}")


In [ ]:
# Langkah 4 - Training dan Evaluasi Model Naive Bayes

# 1. Multinomial Naive Bayes (menggunakan penskalaan non-negatif MinMaxScaler)
from sklearn.preprocessing import MinMaxScaler
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

scaler_mnb = MinMaxScaler()
X_train_mnb = scaler_mnb.fit_transform(X_train)
X_test_mnb = scaler_mnb.transform(X_test)

mnb = MultinomialNB()
mnb.fit(X_train_mnb, y_train)

y_pred_train_mnb = mnb.predict(X_train_mnb)
y_pred_test_mnb = mnb.predict(X_test_mnb)

print("=== HASIL MULTINOMIAL NAIVE BAYES ===")
print(f"Akurasi Data Train : {accuracy_score(y_train, y_pred_train_mnb):.4f}")
print(f"Akurasi Data Test  : {accuracy_score(y_test, y_pred_test_mnb):.4f}")

# 2. Gaussian Naive Bayes (representatif untuk fitur numerik berkelanjutan)
from sklearn.naive_bayes import GaussianNB

gnb = GaussianNB()
gnb.fit(X_train, y_train)

y_pred_train_gnb = gnb.predict(X_train)
y_pred_test_gnb = gnb.predict(X_test)

print("\n=== HASIL GAUSSIAN NAIVE BAYES ===")
print(f"Akurasi Data Train : {accuracy_score(y_train, y_pred_train_gnb):.4f}")
print(f"Akurasi Data Test  : {accuracy_score(y_test, y_pred_test_gnb):.4f}")

print("\nClassification Report (GaussianNB - Test Set):")
print(classification_report(y_test, y_pred_test_gnb, target_names=le_target.classes_))
